In [33]:
faq=[
  {"id": "F01", "q": "Which areas do you deliver to?", "a": "We deliver across Abeokuta, Sagamu, Ijebu-Ode and the Lagos-Ogun corridor. Delivery days depend on your zone."},
  {"id": "F02", "q": "What are your delivery days and times?", "a": "Abeokuta deliveries run Monday to Saturday, 8am to 6pm. Sagamu and Ijebu-Ode deliver Tuesday, Thursday and Saturday."},
  {"id": "F03", "q": "How much does delivery cost?", "a": "Delivery is ₦1,500 inside Abeokuta and ₦2,500 to Sagamu and Ijebu-Ode. Free delivery on orders above ₦25,000."},
  {"id": "F04", "q": "Can I pause or skip my subscription?", "a": "Yes. You can pause or skip a weekly box up to 48 hours before your delivery day from your account, or by contacting support."},
  {"id": "F05", "q": "How do I cancel my subscription?", "a": "You can cancel anytime from your account settings. Cancellations take effect at the end of your current billing week."},
  {"id": "F06", "q": "What is your refund policy?", "a": "If an item arrives damaged or missing, report it within 24 hours with a photo and we will replace or refund it. Refunds are processed by our support team."},
  {"id": "F07", "q": "What payment methods do you accept?", "a": "We accept bank transfer, debit cards, and USSD. Payment is confirmed before your box is dispatched."},
  {"id": "F08", "q": "Do you have vegetarian or vegan options?", "a": "Yes. Our menu includes vegetarian and vegan kits each week; filter by diet on the menu page."},
  {"id": "F09", "q": "How do I handle allergens or dietary needs?", "a": "Each recipe lists allergens. If you have a serious allergy, please confirm with our human support team before ordering — do not rely on the chatbot alone."},
  {"id": "F10", "q": "How are the meal kits packaged?", "a": "Fresh ingredients are insulated and portioned per recipe, with a recipe card. Keep refrigerated and cook within 3 days."},
  {"id": "F11", "q": "Can I change my delivery address?", "a": "Yes, update it in your account up to 48 hours before your delivery day."},
  {"id": "F12", "q": "How do I contact human support?", "a": "WhatsApp or call our team on +234-XXX-XXX-XXXX, Monday to Saturday, 8am to 6pm, or email support@itanfoods.example."}
]

In [34]:
import re
from difflib import SequenceMatcher

def _tokens(s: str) -> set[str]:
    return set(re.findall(r'\w+', s.lower()))

def retrieve_faq(user_text: str, faq: list[dict], top_k:int = 3) -> list[dict]:
    user_tokens = _tokens(user_text)
    scored =[]
    for item in faq:
        item_tokens = _tokens(item['q'] + " " + item['a'])
        overlap = len(user_tokens & item_tokens)
        ratio = SequenceMatcher(None, user_text.lower(), 
        item['q'].lower()).ratio()
        scored.append((overlap + ratio, item))
    scored.sort(key=lambda x: -x[0])
    return [it for _, it in scored[:top_k] if _> 0.3]


# DEVELOP AI APPLICATION

In [35]:
from dotenv import load_dotenv
load_dotenv()
import os
import gradio as gr

print("Key loaded:", bool(os.getenv("GROQ_API_KEY")))

Key loaded: True


In [36]:
#  Pick one provider
import os

PROVIDER ='groq'
MODEL = 'openai/gpt-oss-20b'
MAX_INPUT_CHARS = 2000
API_KEY_ENV ='GROQ_API_KEY'



In [37]:
# Cell 2: single LLM call function
def ask_llm(user_text: str) -> str:
    """Send user_text to the configured LLM. Returns text or raises RuntimeError."""
    key = os.getenv(API_KEY_ENV)
    if not key:
        raise RuntimeError(f"Missing API key. Set env var {API_KEY_ENV} before running.")
    
    hits = retrieve_faq(user_text, faq)
    if hits:
        context = '\n'.join(f"- Q: {h['q']}\n A: {h['a']}" for h in hits)
        system_msg = (
            "You are a customer-support assistant for Itan Foods meal kits. "
            "Answer ONLY using the context below. If the context does not contain "
            "the answer, reply: 'I don't have that information — please contact "
            "human support.' Do not invent policy, prices, or delivery details.\n\n"
            f"CONTEXT:\n{context}"
        )
    else:
        system_msg = (
            "You are a customer-support assistant for Itan Foods meal kits. "
            "You do not have a matching FAQ for this question. Do not invent "
            "answers — reply: 'I don't have that information — please contact "
            "human support.'"
        )


    from groq import Groq
    client = Groq(api_key=key)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_text},
        ],
        temperature=0.3,
        max_tokens=500,
    )
    return resp.choices[0].message.content

    


In [38]:
# Cell 3: user input → LLM → response, with validation + error handling
def get_valid_input() -> str:
    while True:
        text = input("You: ").strip()
        if not text:
            print("[!] Input was empty. Try again.\n")
            continue
        if len(text) > MAX_INPUT_CHARS:
            print(f"[!] Input too long ({len(text)} chars). Max {MAX_INPUT_CHARS}. Try again.\n")
            continue
        if text.lower() in {"quit", "exit"}:
            return "__QUIT__"
        return text

def main():
    print("Barebone LLM app. Type 'quit' to exit.\n")
    while True:
        user_text = get_valid_input()
        if user_text == "__QUIT__":
            print("Bye.")
            break
        try:
            response = ask_llm(user_text)
            print(f"Assistant: {response}\n")
        except RuntimeError as e:
            print(f"[config error] {e}\n")
        except Exception as e:
            # network, rate limit, bad model id, etc.
            print(f"[request failed] {type(e).__name__}: {e}\n")

main()

Barebone LLM app. Type 'quit' to exit.

Bye.


In [39]:
# app.py — barebone AI application
# Run: python app.py
# Requires: pip install gradio groq python-dotenv




def chat_fn(message: str, history: list) -> str:
    """Gradio ChatInterface callback. history is list of [user, bot] pairs."""
    # Validation
    if not message or not message.strip():
        return "[!] Empty input. Please type something."
    if len(message) > MAX_INPUT_CHARS:
        return f"[!] Input too long ({len(message)} chars). Max {MAX_INPUT_CHARS}."

    # Config check
    if not os.getenv('GROQ_API_KEY'):
        return "[config error] GROQ_API_KEY not set. Check your .env file."

    # delegate
    try:
        return ask_llm(message)
    except Exception as e:
        return f"[request failed] {type(e). _name_}: {e}"


# Gradio ChatInterface — no custom Blocks needed
demo = gr.ChatInterface(
    fn=chat_fn,
    title="Customer Support Assistant (Barebone)",
    description="Groq-powered. Type a support question. Max 2000 chars.",
    examples=[
        "How do I reset my password?",
        "What is your refund policy?",
        "Where is my order?",
    ],
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [40]:
import inspect
print('Chat_fn Signature:', inspect.signature(chat_fn))
print('ask_llm Signature:', inspect.signature(ask_llm))

Chat_fn Signature: (message: str, history: list) -> str
ask_llm Signature: (user_text: str) -> str


In [41]:
print(ask_llm('What is your refund policy?'))

If an item arrives damaged or missing, report it within 24 hours with a photo and we will replace or refund it. Refunds are processed by our support team.
